# 03 | Modeling
## Uplift Modeling on the Criteo Uplift Dataset

**Author:** Nguyen (Sue) Huynh

**Last Updated:** June 2026

---

**Objectives**

This notebook implements and evaluates heterogeneous treatment effect estimators on the preprocessed Criteo Uplift dataset. We progressively build T-Learner, S-Learner, X-Learner, and Causal Forest, validating each against the empirical average treatment effect before comparing performance via Qini curves and AUUC in the evaluation notebook.

## 1. Setup and Data Loading

We load the preprocessed train, validation, and test splits from `processed/`, along with the precomputed `scale_pos_weight` values for visit and conversion outcomes saved during preprocessing. The test set is intentionally not used at this stage and is reserved for final, unbiased evaluation of the selected model.

In [4]:
import json
import joblib

import numpy as np
import pandas as pd

from config.config import Config
from src.models.t_learner import TLearner
from src.models.x_learner import XLearner
from src.models.s_learner import SLearner
from src.models.causal_forest import CausalForest

In [2]:
cfg = Config()
with open(cfg.data_dir / "scale_pos_weights.json", "r") as f:
    weights = json.load(f)

scale_pos_weight_visit = weights["scale_pos_weight_visit"]
scale_pos_weight_conv = weights["scale_pos_weight_conv"]

In [3]:
df_train = pd.read_parquet("../processed/train.parquet")
df_val = pd.read_parquet("../processed/val.parquet")

X_train, t_train, y_train, y_secondary_train = df_train[cfg.feature_cols], df_train[cfg.treatment_col], df_train[cfg.outcome_col], df_train[cfg.secondary_outcome_col]
X_val, t_val, y_val, y_secondary_val = df_val[cfg.feature_cols], df_val[cfg.treatment_col], df_val[cfg.outcome_col], df_val[cfg.secondary_outcome_col]

## 2. T-Learner: Baseline Meta-Learner

We begin with T-Learner as our baseline heterogeneous treatment effect estimator. T-Learner fits two independent XGBoost classifiers, one on the treated population and one on the control population, estimating individual uplift as the difference in their predicted probabilities.

### 2.1 Fit and Predict

The T-Learner is fit on the training set using `scale_pos_weight_visit` to address outcome class imbalance, then used to predict uplift on the validation set.

In [4]:
tlearner = TLearner(name="t-learner", scale_pos_weight=scale_pos_weight_visit)
tlearner.fit(X_train, t_train, y_train)
t_tauhat = tlearner.predict_uplift(X_val)
t_tauhat.mean()

np.float32(0.024270196)

### 2.2 Sanity Check Against Empirical ATE

To validate the model produces sensible estimates, we compare the average predicted uplift on the validation set against the empirical Average Treatment Effect (ATE) computed directly from the **validation** data:

In [5]:
ate = df_val[df_val["treatment"] == 1]["visit"].mean() - df_val[df_val["treatment"] == 0]["visit"].mean()
ate

np.float64(0.010560669339153839)

`ATE = mean(visit | T=1) - mean(visit | T=0) ≈ 0.0106`

`Average predicted tau_hat (T-Learner) ≈ 0.024`

### 2.3 Diagnosis: T-Learner Bias Under Group Imbalance

The predicted average uplift overestimates the empirical ATE by a meaningful margin. We attribute this to a known limitation of T-Learner: because the treatment and control models are fit entirely independently, no information is shared between them. Given the Criteo dataset's treatment/control imbalance (approximately 85 percent treated, 15 percent control) combined with a rare outcome (approximately 4.7 percent visit rate), the control model is trained on substantially less signal than the treatment model. This asymmetry in data volume introduces instability into the control model's predictions, biasing the resulting uplift estimate.

## 3. S-Learner

S-Learner fits a single model on the entire training set, with treatment included as a feature alongside the covariates. To estimate uplift, two counterfactual predictions are generated for each unit using the same feature values: one with treatment set to 1, one with treatment set to 0. The predicted uplift is the difference between these two probability estimates.

### 3.1 Fit and Predict

The S-Learner was fit using `scale_pos_weight_visit` and evaluated on the validation set, following the same pattern established for T-Learner.

In [6]:
slearner = SLearner(name="s-learner", scale_pos_weight=scale_pos_weight_visit)
slearner.fit(X_train, t_train, y_train)
s_tauhat = slearner.predict_uplift(X_val)
s_tauhat.mean()

np.float32(0.017590078)

### 3.2 Sanity Check Against Empirical ATE
`ATE = mean(visit | T=1) - mean(visit | T=0) ≈ 0.0106`

`Average predicted tau_hat (T-Learner) ≈ 0.0175`

The S-Learner's average predicted uplift is closer to the empirical ATE than T-Learner's (0.0243), though still somewhat overestimated. Unlike T-Learner, S-Learner shares information across treatment and control groups through a single model, which may partially explain the reduced bias.

## 4. X-Learner

X-Learner addresses a key limitation of T-Learner: when treatment and control groups are imbalanced in size, T-Learner's independently fit models can produce unstable and biased uplift estimates. X-Learner corrects for this through a cross-imputation procedure, combining five fitted components:

1. `model_c`, `model_t`: stage 1 response models, fit separately on control and treatment groups (identical to T-Learner)
2. `model_c_cate`, `model_t_cate`: stage 2 regression models, fit on cross-imputed treatment effects
3. `g`: a propensity model estimating P(treatment=1 | X), used to weight the final combination

The cross-imputed targets are defined as:
> `D1 = y_treatment - mu0_hat(X_treatment)`

> `D0 = mu1_hat(X_control) - y_control`

where `mu0_hat` and `mu1_hat` are cross-group predictions from the stage 1 models. The final uplift estimate combines both stage 2 models, weighted by the propensity score:

> `tau_hat(x) = g(x) * model_c_cate(x) + (1 - g(x)) * model_t_cate(x)`

### 4.1 Diagnosing an Initial Calibration Bug

An initial implementation produced an average predicted uplift of 0.1247, far exceeding the empirical ATE of 0.0106. Inspecting intermediate values revealed the source: `mu0_hat` and `mu1_hat` averaged approximately 0.19 and 0.20 respectively, roughly four times higher than the true visit rate of approximately 4.7%.

This inflation was traced to `scale_pos_weight`, which was applied to the stage 1 classifiers `model_c` and `model_t`. While `scale_pos_weight` is necessary for handling class imbalance in T-Learner and S-Learner, it introduces a known side effect of inflating predicted probabilities away from their true calibrated values. In T-Learner and S-Learner, this inflation is largely symmetric across the models being differenced and partially cancels out in the final uplift calculation. In X-Learner, however, `mu0_hat` and `mu1_hat` are subtracted directly against true binary outcomes to construct the cross-imputed targets `D1` and `D0`, with no second inflated model to cancel against. The inflation therefore propagates directly into the regression targets and biases the final uplift estimate substantially upward.

### 4.2 Resolution

`scale_pos_weight` was removed from the stage 1 models (`model_c`, `model_t`) used specifically within X-Learner. The constructor parameter is retained for interface consistency with T-Learner and S-Learner but is intentionally not applied internally, with this behavior documented directly in the class and verified with a dedicated unit test.

In [7]:
xlearner = XLearner(name="x-learner")
xlearner.fit(X_train, t_train, y_train)
x_tauhat = xlearner.predict_uplift(X_val)
x_tauhat.mean()

np.float64(0.007747098955537449)

### 4.3 Sanity Check Against Empirical ATE
`ATE = mean(visit | T=1) - mean(visit | T=0) ≈ 0.0106`

`Average predicted tau_hat (T-Learner) ≈ 0.0077`

| Model | Average Predicted Uplift | Empirical ATE |
|---|---|---|
| T-Learner | 0.0243 | 0.0106 |
| S-Learner | 0.0176 | 0.0106 |
| X-Learner | 0.0077 | 0.0106 |

Following the fix, X-Learner produces the closest average uplift estimate to the empirical ATE among all three meta-learners implemented so far, consistent with its design motivation of correcting for treatment and control group imbalance.

## 5. Causal Forest

Causal Forest represents a fundamentally different approach to heterogeneous treatment effect estimation compared to the meta-learners implemented above. Rather than combining off-the-shelf classifiers through clever data splitting, Causal Forest is a generalized random forest with a custom tree-splitting criterion: trees split to directly maximize heterogeneity in treatment effects between leaves, using an "honest splitting" procedure that uses separate subsamples for determining splits versus estimating leaf-level effects.

### 5.1 Implementation Choice: EconML over From-Scratch

Unlike T-Learner, S-Learner, and X-Learner, which were implemented from scratch using XGBoost as the base learner, Causal Forest was implemented using EconML's `CausalForestDML`. This was a deliberate engineering decision rather than a shortcut. Implementing a valid honest-splitting causal tree from scratch, with the accompanying asymptotic theory for unbiased variance estimation, represents a substantially different and more specialized engineering effort than combining standard classifiers. EconML's implementation is well-tested and widely used in production causal inference settings, making it the appropriate tool for this specific algorithm.

### 5.2 Architecture

`CausalForestDML` requires two nuisance models in addition to the causal forest itself:

- `model_y`: estimates `E[Y | X]`, used for outcome residualization
- `model_t`: estimates `E[T | X]`, used for treatment residualization

Both outcome and treatment in this dataset are binary, so `discrete_outcome=True` and `discrete_treatment=True` were set, requiring both nuisance models to be classifiers (`RandomForestClassifier`) rather than regressors, since `predict_proba` is needed for residualization in the discrete case.

### 5.3 Hyperparameter Tuning

Two parameters were tuned deliberately based on dataset characteristics rather than arbitrary defaults:

**`max_depth=10`** and **`n_jobs=2`**: Without a depth cap, trees attempting to isolate a rare positive outcome (visit rate approximately 4.7 percent) can grow very deep, consuming substantial memory. Both parameters were necessary to keep training stable on a local machine; without them, the kernel terminated due to memory exhaustion even at sample sizes as small as 100,000 rows.

**`min_samples_leaf=450`**: This value was derived directly from the outcome's class imbalance rather than chosen arbitrarily. With a visit rate of approximately 4.7 percent, a leaf of size 50 (an earlier attempt) contains only about 2.4 expected positive outcomes on average, far too few to support a stable effect estimate. Targeting approximately 20 to 30 expected positive outcomes per leaf for stability requires a minimum leaf size of approximately 20 divided by 0.047, or roughly 425 to 500 samples. This adjustment substantially tightened the range of predicted treatment effects, though it was not the only issue affecting model performance.

In [5]:
causalforest = CausalForest(name="causal-forest")
causalforest.fit(X_train, t_train, y_train)

c:\Users\nguye\uplift-modeling\.venv\Lib\site-packages\econml\dml\causal_forest.py:72: UserWarning: Could not generate out-of-bag predictions on some training data. Consider increasing the number of trees. `ate_` results will take the average of the subset of training data for which out-of-bag predictions where available.
  warn("Could not generate out-of-bag predictions on some training data. "


CausalForest(name='causal-forest', status=fitted)

In [6]:
causalforest.predict_uplift(X_val)

array([4.86896921e-04, 3.19062417e-04, 4.81213593e-04, ...,
       2.80386371e-03, 2.62268118e-04, 8.57525102e-05], shape=(200000,))

> Saved all trained models for Evaluation.

In [ ]:
import joblib

joblib.dump(tlearner, cfg.models_dir / 'tlearner.joblib')
joblib.dump(slearner, cfg.models_dir / 'slearner.joblib')
joblib.dump(xlearner, cfg.models_dir / 'xlearner.joblib')
joblib.dump(causalforest, cfg.models_dir / 'causalforest.joblib')